In [ ]:
import pandas as pd

file_g027 = 'G027_predictions.csv'
file_g028 = 'G028_predictions.csv'  
file_g029 = 'G029_predictions.csv'  

In [ ]:
import pandas as pd

file_g027 = 'G027_predictions.csv'
file_g028 = 'G028_predictions.csv'  
file_g029 = 'G029_predictions.csv'  
df_g027 = pd.read_csv(file_g027)
df_g028 = pd.read_csv(file_g028)
df_g029 = pd.read_csv(file_g029)
df_g027 = df_g027.add_suffix('_g027')
df_g027.rename(columns={'cohort_g027': 'cohort', 'elapsed_months_g027': 'elapsed_months', 'current_month_g027': 'current_month'}, inplace=True)
merged = df_g029.merge(df_g028, on=['cohort', 'elapsed_months', 'current_month'], how='outer', suffixes=('_g029', '_g028'))
merged = merged.merge(df_g027, on=['cohort', 'elapsed_months', 'current_month'], how='outer')
for col in merged.columns:
    if merged[col].dtype in ['int64', 'float64']:
        merged[col].fillna(0, inplace=True)
    elif merged[col].dtype == 'object':
        merged[col].fillna('', inplace=True)
result = pd.DataFrame()
result['cohort'] = merged['cohort']
result['elapsed_months'] = merged['elapsed_months']
result['current_month'] = merged['current_month']

result['game_id'] = (merged['game_id_g027'].fillna('') + '+' +
                     merged['game_id_g028'].fillna('') + '+' +
                     merged['game_id_g029'].fillna(''))
numerical_columns = ['absolute_revenue_actual', 'total_cac', 'absolute_revenue_bad',
                     'absolute_revenue_good', 'absolute_revenue_expected']

for col in numerical_columns:
    result[col] = (merged[f'{col}_g027'].fillna(0) +
                   merged[f'{col}_g028'].fillna(0) +
                   merged[f'{col}_g029'].fillna(0))

result['total_cac'].replace(0, 1e-9, inplace=True)

result['transactional_actual'] = result['absolute_revenue_actual'] / result['total_cac']
result['transactional_bad'] = result['absolute_revenue_bad'] / result['total_cac']
result['transactional_good'] = result['absolute_revenue_good'] / result['total_cac']
result['transactional_expected'] = result['absolute_revenue_expected'] / result['total_cac']


result['cumulative_actual'] = 0
result['cumulative_bad'] = 0
result['cumulative_good'] = 0
result['cumulative_expected'] = 0

for cohort, group in result.groupby('cohort'):
    cumulative_actual = 0
    cumulative_bad = 0
    cumulative_good = 0
    cumulative_expected = 0

    for index, row in group.iterrows():
        cumulative_actual += row['transactional_actual']
        cumulative_bad += row['transactional_bad']
        cumulative_good += row['transactional_good']
        cumulative_expected += row['transactional_expected']

        result.loc[index, 'cumulative_actual'] = cumulative_actual
        result.loc[index, 'cumulative_bad'] = cumulative_bad
        result.loc[index, 'cumulative_good'] = cumulative_good
        result.loc[index, 'cumulative_expected'] = cumulative_expected
result = result[df_g029.columns]
result['game_id'].replace('++G029', 'G029', inplace=True)
output_file = 'G027_G028_G029_combined.csv'
result.to_csv(output_file, index=False)

print(f'Results saved to {output_file}')

df1 = pd.read_csv('G027_G028_G029_combined.csv')
df2 = pd.read_csv('G030_predictions.csv')

final_results = []
for _, row1 in df1.iterrows():
    for _, row2 in df2.iterrows():
        if row1['cohort'] == row2['cohort'] and row1['elapsed_months'] == row2['elapsed_months']:
            if pd.isnull(row1['total_cac']) or pd.isnull(row2['total_cac']) or pd.isnull(row1['absolute_revenue_actual']) or pd.isnull(row2['absolute_revenue_actual']):
                continue
            cac_error = calculate_error_with_zero_handling(row1['total_cac'], row2['total_cac'])
            revenue_error = calculate_error_with_zero_handling(row1['absolute_revenue_actual'], row2['absolute_revenue_actual'])
            match = "Yes" if cac_error <= 2 and revenue_error <= 2 else "No"
            final_results.append({
                'cohort': row1['cohort'],
                'elapsed_months': row1['elapsed_months'],
                'percent_error_ltv_cac': cac_error,
                'percent_error_absolute_revenue_actual': revenue_error,
                'match': match
            })

final_results_df = pd.DataFrame(final_results)
final_results_df.to_csv('30_data_check.csv', index=False)
print("Final cohort error analysis saved.")


In [ ]:
df_g027 = pd.read_csv(file_g027)
df_g028 = pd.read_csv(file_g028)
df_g029 = pd.read_csv(file_g029)

# common_cohorts = set(df_g027['cohort']).intersection(df_g028['cohort']).intersection(df_g029['cohort'])
# df_g027 = df_g027[df_g027['cohort'].isin(common_cohorts)]
# df_g028 = df_g028[df_g028['cohort'].isin(common_cohorts)]
# df_g029 = df_g029[df_g029['cohort'].isin(common_cohorts)]

In [ ]:
df_g027 = df_g027.add_suffix('_g027')
df_g027.rename(columns={'cohort_g027': 'cohort', 'elapsed_months_g027': 'elapsed_months', 'current_month_g027': 'current_month'}, inplace=True)
merged = df_g029.merge(df_g028, on=['cohort', 'elapsed_months', 'current_month'], how='outer', suffixes=('_g029', '_g028'))
merged = merged.merge(df_g027, on=['cohort', 'elapsed_months', 'current_month'], how='outer')


In [ ]:
merged.columns

In [ ]:
for col in merged.columns:
    if merged[col].dtype in ['int64', 'float64']:
        merged[col].fillna(0, inplace=True)
    elif merged[col].dtype == 'object':
        merged[col].fillna('', inplace=True)

In [ ]:
result = pd.DataFrame()
result['cohort'] = merged['cohort']
result['elapsed_months'] = merged['elapsed_months']
result['current_month'] = merged['current_month']

result['game_id'] = (merged['game_id_g027'].fillna('') + '+' +
                     merged['game_id_g028'].fillna('') + '+' +
                     merged['game_id_g029'].fillna(''))


In [ ]:
result

In [ ]:
numerical_columns = ['absolute_revenue_actual', 'total_cac', 'absolute_revenue_bad',
                     'absolute_revenue_good', 'absolute_revenue_expected']

for col in numerical_columns:
    result[col] = (merged[f'{col}_g027'].fillna(0) +
                   merged[f'{col}_g028'].fillna(0) +
                   merged[f'{col}_g029'].fillna(0))

result['total_cac'].replace(0, 1e-9, inplace=True)

result['transactional_actual'] = result['absolute_revenue_actual'] / result['total_cac']
result['transactional_bad'] = result['absolute_revenue_bad'] / result['total_cac']
result['transactional_good'] = result['absolute_revenue_good'] / result['total_cac']
result['transactional_expected'] = result['absolute_revenue_expected'] / result['total_cac']

In [ ]:
result['cumulative_actual'] = 0
result['cumulative_bad'] = 0
result['cumulative_good'] = 0
result['cumulative_expected'] = 0

In [ ]:
for cohort, group in result.groupby('cohort'):
    cumulative_actual = 0
    cumulative_bad = 0
    cumulative_good = 0
    cumulative_expected = 0

    for index, row in group.iterrows():
        cumulative_actual += row['transactional_actual']
        cumulative_bad += row['transactional_bad']
        cumulative_good += row['transactional_good']
        cumulative_expected += row['transactional_expected']

        result.loc[index, 'cumulative_actual'] = cumulative_actual
        result.loc[index, 'cumulative_bad'] = cumulative_bad
        result.loc[index, 'cumulative_good'] = cumulative_good
        result.loc[index, 'cumulative_expected'] = cumulative_expected

In [ ]:
result = result[df_g029.columns]

In [ ]:
result.shape

In [ ]:
result['game_id'].replace('++G029', 'G029', inplace=True)

In [ ]:
output_file = 'G027_G028_G029_combined.csv'
result.to_csv(output_file, index=False)

print(f'Results saved to {output_file}')